Supervised Contrastive Loss (SupervisedContrastive)
Based on the formulation of Supervised Contrastive Learning:
Khosla et al., "Supervised Contrastive Learning", NeurIPS 2020.
Implementation inspired by:
https://github.com/wangz10/contrastive_loss
Modified for integration with the projection head and training pipeline in this project.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    confusion_matrix, classification_report, accuracy_score, cohen_kappa_score, f1_score
)
import scanpy as sc

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams["text.usetex"] = False

In [ ]:
RGC_adata = sc.read_h5ad("#file")

/opt/anaconda3/envs/openai028/lib/python3.10/site-packages/anndata/compat/__init__.py:371: FutureWarning: Moving element from .uns['neighbors']['distances'] to .obsp['distances'].

This is where adjacency matrices should go now.
  warn(


In [ ]:
embedding_hkkeep = np.load("#file")
embedding_hkfilter = np.load("#file")

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu' 

Projection head

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader, random_split
import numpy as np

class ProjectionHead(nn.Module):
    def __init__(self, input_dim = 1536, hidden_dim = 512, output_dim = 128):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim, bias= False)
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.relu = nn.ReLU(inplace=True)
        self.fc2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = self.fc1(x)
        x = self.bn1(x)
        x = self.relu(x)
        z = self.fc2(x)
        z = F.normalize(z, dim=1)
        return z


SupConLoss Defination

In [ ]:
class SupervisedContrastive(nn.Module):
    def __init__(self, temperature=0.07, contrast_mode="all", base_temperature=0.07):
        super().__init__()
        self.temperature = temperature
        self.contrast_mode = contrast_mode
        self.base_temperature = base_temperature

    def forward(self, embeddings, labels=None):
        # check feature shape
        if embeddings.ndim != 3:
            raise ValueError(
                "embeddings must have shape [batch_size, n_views, embedding_dim]"
            )

        device = embeddings.device
        batch_size, n_views, embedding_dim = embeddings.shape

        # [batch_size, n_views, dim] -- [batch_size * n_views, dim]
        contrast_embeddings = embeddings.reshape(batch_size * n_views, embedding_dim)

        # Choose anchor embeddings
        if self.contrast_mode == "all":
            anchor_embeddings = contrast_embeddings
            anchor_batch_size = batch_size * n_views
            anchor_view_count = n_views

        elif self.contrast_mode == "one":
            anchor_embeddings = embeddings[:, 0, :]
            anchor_batch_size = batch_size
            anchor_view_count = 1

        else:
            raise ValueError("contrast_mode must be either 'all' or 'one'.")

        # Build label-based positive-pair matrix at sample level
        if labels is None:
            sample_positive = torch.eye(batch_size, device=device)
        else:
            labels = labels.to(device).reshape(batch_size, 1)
            sample_positive = (labels == labels.T).float()

        # Expand sample-level mask to view-level mask
        positive_mask = sample_positive.repeat(anchor_view_count, n_views)

        # Compute similarity matrix
        similarity = torch.matmul(anchor_embeddings, contrast_embeddings.T)
        similarity = similarity / self.temperature

        # Stabilize logits
        similarity = similarity - similarity.max(dim=1, keepdim=True).values.detach()

        # Remove self-comparisons
        self_comparison_mask = torch.ones(
            anchor_batch_size,
            batch_size * n_views,
            device=device
        )

        diagonal_indices = torch.arange(anchor_batch_size, device=device)
        self_comparison_mask[diagonal_indices, diagonal_indices] = 0.0

        positive_mask = positive_mask * self_comparison_mask

        # Compute log probability
        exp_similarity = torch.exp(similarity) * self_comparison_mask
        denominator = exp_similarity.sum(dim=1, keepdim=True).clamp_min(1e-12)

        log_probability = similarity - torch.log(denominator)

        # Average only over positive pairs
        positive_count = positive_mask.sum(dim=1).clamp_min(1e-12)
        mean_log_probability = (positive_mask * log_probability).sum(dim=1) / positive_count

        loss = -mean_log_probability
        loss = loss * (self.temperature / self.base_temperature)

        return loss.mean()
    
criterion = SupervisedContrastive(temperature=0.07, contrast_mode='all').to(device) # criterion defination
   

extract label

In [ ]:
import numpy as np
import torch

cluster = RGC_adata.obs['cell_cluster'].astype(str).to_numpy()

uniq = {c:i for i, c in enumerate(np.unique(cluster))}
labels_np = np.array([uniq[c] for c in cluster], dtype = np.int64)

labels = torch.tensor(labels_np, device = device)


early stopping

In [ ]:
import torch

# device = 'cuda' if torch.cuda.is_available() else 'cpu' 
projector = ProjectionHead(input_dim=1536, hidden_dim=512, output_dim=128)
optimizer = torch.optim.Adam(projector.parameters(), lr = 1e-3)

#tensor
raw_tensor = torch.as_tensor(embedding_hkkeep, dtype=torch.float32).to(device)
hk_tensor = torch.as_tensor(embedding_hkfilter, dtype=torch.float32).to(device)
y = torch.tensor(labels_np, dtype=torch.long)

# Divide batch and loop
N = raw_tensor.size(0)
val_ratio = 0.1
n_val = int(N * val_ratio)
n_train = N - n_val
full_ds = TensorDataset(raw_tensor, hk_tensor, y)
train_ds, val_ds = random_split(full_ds, [n_train, n_val],
                                generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=256, shuffle=False, drop_last=False)

In [ ]:
projector = projector.to(device).float()
optimizer = torch.optim.Adam(projector.parameters(), lr=1e-3, weight_decay=1e-6)

max_epochs  = 50             # fix epochs maximum
patience    = 7              # early stopping patience
best_val    = float('inf')
no_improve  = 0
tol         = 1e-4           

for epoch in range(1, max_epochs + 1):
    projector.train()
    train_loss = 0.0
    for bx1, bx2, by in train_loader:
        bx1, bx2, by = bx1.to(device), bx2.to(device), by.to(device)

        z1 = F.normalize(projector(bx1), dim=1)
        z2 = F.normalize(projector(bx2), dim=1)
        feats = torch.stack([z1, z2], dim=1)  # [bsz, 2, dim]

        loss = criterion(feats, labels=by) # Use SupervisedContrastive
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= max(1, len(train_loader))


    # Evaluation
    projector.eval()
    val_loss = 0.0
    with torch.no_grad():
        for bx1, bx2, by in val_loader:
            bx1, bx2, by = bx1.to(device), bx2.to(device), by.to(device)
            z1 = F.normalize(projector(bx1), dim=1)
            z2 = F.normalize(projector(bx2), dim=1)
            feats = torch.stack([z1, z2], dim=1)
            loss = criterion(feats, labels=by)
            val_loss += loss.item()
    val_loss /= max(1, len(val_loader))


    print(f"Epoch {epoch:02d} | train {train_loss:.4f} | val {val_loss:.4f}")

    # Early stopping
    if best_val - val_loss > tol:   
        best_val = val_loss
        no_improve = 0
        
    else:
        no_improve += 1
        if no_improve >= patience:
            print(f"Early stopping at epoch {epoch} (no improve for {patience} epochs).")
            break

Epoch 01 | train 6.2367 | val 4.7807
Epoch 02 | train 6.2369 | val 4.7809
Epoch 03 | train 6.2367 | val 4.7811
Epoch 04 | train 6.2370 | val 4.7801
Epoch 05 | train 6.2365 | val 4.7801
Epoch 06 | train 6.2366 | val 4.7802
Epoch 07 | train 6.2366 | val 4.7803
Epoch 08 | train 6.2366 | val 4.7798
Epoch 09 | train 6.2365 | val 4.7798
Epoch 10 | train 6.2366 | val 4.7801
Epoch 11 | train 6.2365 | val 4.7800
Epoch 12 | train 6.2365 | val 4.7799
Epoch 13 | train 6.2366 | val 4.7798
Epoch 14 | train 6.2365 | val 4.7797
Epoch 15 | train 6.2364 | val 4.7797
Epoch 16 | train 6.2366 | val 4.7796
Epoch 17 | train 6.2365 | val 4.7795
Epoch 18 | train 6.2365 | val 4.7792
Epoch 19 | train 6.2365 | val 4.7790
Epoch 20 | train 6.2365 | val 4.7791
Epoch 21 | train 6.2364 | val 4.7793
Epoch 22 | train 6.2364 | val 4.7796
Epoch 23 | train 6.2365 | val 4.7797
Epoch 24 | train 6.2364 | val 4.7801
Epoch 25 | train 6.2367 | val 4.7800
Epoch 26 | train 6.2365 | val 4.7798
Early stopping at epoch 26 (no improve

In [ ]:
embedding_hkkeep = torch.from_numpy(embedding_hkkeep).float().to(device)
embedding_hkfilter  = torch.from_numpy(embedding_hkfilter).float().to(device)

In [ ]:
with torch.no_grad():
    contrastive_embeddings = projector(raw_tensor).cpu().numpy()

In [ ]:
np.save("", contrastive_embeddings)

In [ ]:
# Upload contrastive learning npy file
contrastive_embeddings = np.load("#file")

KNN classifier

In [ ]:
cell_type_map = {
    "aff": "α FF",
    "afr": "α FR",
    "asf": "α SF",
    "g": "γ"
    }

MN_adata.obs["cell_types"] = MN_adata.obs["cell_type"].map(cell_type_map)

x = contrastive_embeddings
y = MN_adata.obs["cell_type"].values


In [ ]:
cell_type_map = {
    "43_AlphaONS": "C43",
    "28_FmidiOFF": "C28",
    "33_M1": "C33",
    "36_Novel": "C36",
    "22_M5": "C22",
    "35_Novel": "C35",
    "31_M2": "C31",
    "37_Novel": "C37"
    }

RGC_adata.obs["cell_types"] = RGC_adata.obs["cell_cluster"].map(cell_type_map).fillna("unknown")

x = contrastive_embeddings
y = RGC_adata.obs["cell_cluster"].values

selected_cell_types = ['C43','C28','C33','C36',"C22","C35","C31","C37"]


In [79]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, stratify=y, random_state=1)
# 20% sample distribute to test data, 80% sample distribute to train data.

knn = KNeighborsClassifier(n_neighbors=10, metric="cosine")
knn.fit(x_train, y_train) 

y_pred = knn.predict(x_test)

accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='weighted')  # 'weighted' accounts for class imbalance
kappa = cohen_kappa_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.4f}")
print(f"F1 Score (weighted): {f1:.4f}")
print(f"Cohen's Kappa: {kappa:.4f}")

Model Accuracy: 0.9180
F1 Score (weighted): 0.9181
Cohen's Kappa: 0.8975


Confusion Matrix

In [ ]:
import pandas as pd
import numpy as np
import scanpy as sc
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, f1_score, cohen_kappa_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

In [ ]:
# Close Latex rendering mode
import matplotlib.pyplot as plt
plt.rcParams["text.usetex"] = False

In [ ]:
label_dict = {
    "afr": "α FR MN",
    "aff": "α FF MN",
    "asf": "α SF MN",
    "g": "γ MN"
}

In [ ]:
label_dict = {
    "43_AlphaONS": "C43",
    "28_FmidiOFF": "C28",
    "33_M1": "C33",
    "36_Novel": "C36",
    "22_M5": "C22",
    "35_Novel": "C35"
    }

In [ ]:
label_order = np.unique(np.concatenate([y_test, y_pred]))

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=label_order)
cm_normalized = cm.astype("float") / cm.sum(axis=1, keepdims=True)

# Define class labels dynamically from y_test
labels = np.unique(np.concatenate([y_test, y_pred]))
pretty_labels = [label_dict[l] for l in label_order]

plt.rcParams["font.family"] = "Arial"
plt.rcParams["font.style"] = "normal"
plt.rcParams["axes.titleweight"] = "normal"
plt.rcParams["axes.labelweight"] = "normal"

# Plot the normalized confusion matrix
plt.figure(figsize=(9, 8))
sns.heatmap(cm_normalized, annot=True, fmt=".2f", cmap="Blues", xticklabels=pretty_labels, yticklabels=pretty_labels, cbar_kws={"shrink": 0.85}, vmin=0, vmax=1, annot_kws={"size":15, "weight":"normal"})
ax = plt.gca()

ax.set_xticklabels(pretty_labels, fontsize=20, fontweight="normal", fontfamily="Arial", rotation=0, color="black")
ax.set_yticklabels(pretty_labels, fontsize=20, fontweight="normal", fontfamily="Arial", rotation=0, color="black")
for spine in ax.spines.values(): spine.set_visible(True), spine.set_edgecolor("black"), spine.set_linewidth(1)
cbar = ax.collections[0].colorbar
ticks = list(cbar.get_ticks())
ticks = sorted(set(ticks + [0,1]))

cbar.set_ticks(ticks)
cbar.outline.set_edgecolor("black")
cbar.outline.set_linewidth(1)
cbar.ax.tick_params(labelsize=15, labelfontfamily="Arial", color="black")
for label in cbar.ax.get_yticklabels():label.set_fontweight("normal")

plt.xlabel("Predicted Labels", fontsize=20, fontweight="normal", fontfamily="Arial", color="black")
plt.ylabel("True Labels",fontsize=20, fontweight="normal", fontfamily="Arial", color="black")
plt.title(f"Normalized Confusion Matrix for KNN Classifier\n With Embedding, without Contrastive Learning \nModel: text-embedding-ada-002\nAccuracy: {accuracy:.4f} | F1: {f1:.4f} | Kappa: {kappa:.4f}",
          fontsize=17, fontweight="normal", fontfamily="Arial", color="black")
plt.tight_layout()

plt.show()

# Print classification report
labels = np.unique(np.concatenate([y_test, y_pred]))
print("Classification Report:")
print(classification_report(y_test, y_pred, labels=label_order, target_names=label_dict))

Umap

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import umap
from sklearn.manifold import TSNE
import scanpy as sc
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

In [ ]:
from matplotlib.ticker import MultipleLocator

In [ ]:
# Upload contrastive learning processed npy file
contrastive_embeddings = np.load("#file")

In [ ]:
label_dict_umap = {
    "afr": "α FR MN",
    "aff": "α FF MN",
    "asf": "α SF MN",
    "g": "γ MN"
}

In [ ]:
label_dict_umap = {
    "43_AlphaONS":"C43_AlphaONS",
    "28_FmidiOFF":"C28_FmidiOFF",
    "36_Novel":"C36_Novel",
    "33_M1":"C33_M1",
    "22_M5":"C22_M5",
    "35_Novel":"C35_Novel"
}

In [ ]:
# Load your embeddings and labels
labels = RGC_adata.obs['cell_type'].values  

# UMAP
reducer_umap = umap.UMAP(random_state=1)
embedding_umap = reducer_umap.fit_transform(contrastive_embeddings)

plt.rcParams["font.family"] = "Arial"
plt.figure(figsize=(10, 8))
plt.style.use('default')

# color_map = {"a": "#ff7f0e","g": "#1f77b4"}
              
for label in np.unique(labels):
    idx = labels == label
    plt.scatter(
        embedding_umap[idx, 0], 
        embedding_umap[idx, 1], 
        label=label_dict_umap.get(str(label), f"Cluster{label}"),alpha=0.8, zorder=2)
        # color=color_map.get(label))
        #label=label, s=15, alpha=0.8, zorder=2)

plt.grid(True, linestyle='--', linewidth=1.0, alpha=1.0, zorder=0)

ax = plt.gca()
ax.xaxis.set_major_locator(MultipleLocator(2.5))   
ax.yaxis.set_major_locator(MultipleLocator(2.5))
ax.tick_params(axis='both', labelsize=20)
for spine in ax.spines.values():
    spine.set_linewidth(1)
    spine.set_color("black")
for label in ax.get_xticklabels():
    label.set_fontfamily("Arial")
for label in ax.get_yticklabels():
    label.set_fontfamily("Arial")


# Model: text-embedding-002
plt.title("UMAP of KNN Classifier\nWith Embedding and Contrastive Learning\nModel: text-embedding-ada-002",fontsize=20, fontfamily="Arial")
plt.legend(loc="upper right", bbox_to_anchor=(1, 1),prop={"family": "Arial", "size":15})
plt.xlabel("UMAP1",fontsize=20, fontfamily="Arial")
plt.ylabel("UMAP2",fontsize=20)
plt.grid(True)
plt.show()

sil_score = silhouette_score(contrastive_embeddings, labels, metric='cosine', random_state =1)
print(f"Silhouette Score (LLM embeddings): {sil_score:.4f}")
ch_score = calinski_harabasz_score(contrastive_embeddings, labels)
print(f"Calinski-Harabasz Score (LLM embeddings): {ch_score:.2f}")
db_score = davies_bouldin_score(contrastive_embeddings, labels)
print(f"Davies-Bouldin Score (LLM embeddings): {db_score:.2f}")